In [29]:
import os
import pandas as pd
from tqdm import tqdm
import numpy as np

In [17]:
path = 'token_data.tsv'
df = pd.read_csv(path, sep='\t')

In [18]:
len(df), df.columns.tolist()

(23514,
 ['text',
  'lemma',
  'parts_lemma',
  'pos',
  'parts_pos',
  'token_count',
  'token_pct',
  'group_count',
  'group_pct',
  'xpos',
  'deprel',
  'feats',
  'token_hash',
  'group_hash',
  'sentences'])

In [19]:
import uuid
# fix group hashes
# Group by lemma so cliticized forms land in the base verb bucket
def get_group_hash(lemma, pos):
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{lemma}-{pos}"))
df['group_hash'] = df.apply(lambda row: get_group_hash(row['lemma'], row['pos']), axis=1)

In [20]:
df['group'] = df.apply(lambda row: f"lemma={row['lemma']}, pos={row['pos']}", axis=1)
print(len(df['group'].unique()))
print(len(df['group_hash'].unique()))

11937
11937


In [21]:
sentence_df = pd.read_csv('sentence_dataframe.tsv', sep='\t')
len(sentence_df), sentence_df.columns

(624335, Index(['text_it', 'text_en', 'hash'], dtype='object'))

In [22]:
sentence_df.set_index('hash', inplace=True, drop=True)

In [23]:
group_pct_dict = {}
group_hash_to_str = {}
group_hashes = list(df['group_hash'].unique())
for group_hash in tqdm(group_hashes, total=len(group_hashes)):
    row = df[df['group_hash'] == group_hash].iloc[0]
    group_pct_dict[group_hash] = row['group_pct']
    group_hash_to_str[group_hash] = row['group']

100%|██████████| 11937/11937 [00:07<00:00, 1616.71it/s]


In [24]:
all_groups = list(df['group_hash'].unique())
all_groups = sorted(all_groups, key=lambda x: group_pct_dict[x], reverse=True)
[group_hash_to_str[hash] for hash in all_groups[:10]]

['lemma=essere, pos=AUX',
 'lemma=essere, pos=VERB',
 'lemma=essere, pos=NOUN',
 'lemma=essere, pos=PRON',
 'lemma=il, pos=DET',
 'lemma=il, pos=AUX',
 'lemma=il, pos=ADV',
 'lemma=di, pos=ADP',
 'lemma=di, pos=DET',
 'lemma=avere, pos=AUX']

In [25]:
N = 200
top_groups = all_groups[:N]
reduced_df = df[df['group_hash'].isin(top_groups)]
len(reduced_df)

1706

In [26]:
reduced_df.columns

Index(['text', 'lemma', 'parts_lemma', 'pos', 'parts_pos', 'token_count',
       'token_pct', 'group_count', 'group_pct', 'xpos', 'deprel', 'feats',
       'token_hash', 'group_hash', 'sentences', 'group'],
      dtype='object')

In [30]:
reduced_df[np.logical_and(
    reduced_df['pos'] == 'ADP',
    reduced_df['lemma'] == 'da'
)]

,text,lemma,parts_lemma,pos,parts_pos,token_count,token_pct,group_count,group_pct,xpos,deprel,feats,token_hash,group_hash,sentences,group
376,da,da,da,ADP,ADP,3523,0.005155,5495,0.008041,E,case,NaN,b7a5f726-e2a2-52aa-83c5-6e6018a53df6,af61b489-9773-510d-bbeb-5c909fbf4068,"['52d70ad4-fe6f-5fa9-924e-03072e4fc97b', '3002...","lemma=da, pos=ADP"
377,dalla,da,da+il,ADP,ADP+DET,733,0.001073,5495,0.008041,E,case,Definite=Def|Gender=Fem|Number=Sing|PronType=Art,973fcb75-1792-55b6-bb8f-5e39f8f13874,af61b489-9773-510d-bbeb-5c909fbf4068,"['514f7492-06c7-5a85-9d9e-8cec43e5bbab', '6523...","lemma=da, pos=ADP"
378,dal,da,da+il,ADP,ADP+DET,614,0.000898,5495,0.008041,E,case,Definite=Def|Gender=Masc|Number=Sing|PronType=Art,ccf13769-4049-5494-89b9-997961cab71d,af61b489-9773-510d-bbeb-5c909fbf4068,"['99b5290c-5227-5605-b08c-9ee19cdfe4d4', '50a3...","lemma=da, pos=ADP"
379,dall',da,da+il,ADP,ADP+DET,304,0.000445,5495,0.008041,E,case,Definite=Def|Number=Sing|PronType=Art,9bd04a01-dca3-56f0-886b-f6c7a26aa864,af61b489-9773-510d-bbeb-5c909fbf4068,"['513ef2ee-5b1a-5317-993d-f70a576e8cbc', '0e0b...","lemma=da, pos=ADP"
380,dai,da,da+il,ADP,ADP+DET,121,0.000177,5495,0.008041,E,case,Definite=Def|Gender=Masc|Number=Plur|PronType=Art,75bea919-e7ec-50c1-95df-cba1abf1233f,af61b489-9773-510d-bbeb-5c909fbf4068,"['12fa2154-9c9d-5e95-9f4a-715f9b45c172', 'fbf2...","lemma=da, pos=ADP"
381,dalle,da,da+il,ADP,ADP+DET,100,0.000146,5495,0.008041,E,case,Definite=Def|Gender=Fem|Number=Plur|PronType=Art,a22a7c58-958b-5c20-b869-e3e5f4a469c2,af61b489-9773-510d-bbeb-5c909fbf4068,"['cc3d2614-0e6e-5b8e-ae38-81021ffbb3dc', '640d...","lemma=da, pos=ADP"
382,dagli,da,da+il,ADP,ADP+DET,47,0.000069,5495,0.008041,E,case,Definite=Def|Gender=Masc|Number=Plur|PronType=Art,a97d6b9f-0bdf-5336-ba1f-a9db936ec3e3,af61b489-9773-510d-bbeb-5c909fbf4068,"['64a4151a-91b2-5da3-aae2-1dbe4b1f525c', '9629...","lemma=da, pos=ADP"
383,dallo,da,da+il,ADP,ADP+DET,39,0.000057,5495,0.008041,E,case,Definite=Def|Gender=Masc|Number=Sing|PronType=Art,d0560088-a98f-5254-85d1-d7b8ab983aea,af61b489-9773-510d-bbeb-5c909fbf4068,"['1f9be566-158c-5804-829a-e6416aa7083b', '9097...","lemma=da, pos=ADP"
384,d',da,da,ADP,ADP,9,0.000013,5495,0.008041,E,case,NaN,6cf0d48c-9026-53a3-b2cf-5b2aa83453db,af61b489-9773-510d-bbeb-5c909fbf4068,"['b39d0abb-66f2-5bac-8c18-6321a5c9d898', '8887...","lemma=da, pos=ADP"
385,dategli,da,da+te+gli,ADP,ADP+PRON+PRON,1,0.000001,5495,0.008041,E,case,Clitic=Yes|Number=Sing|Person=2|PronType=Prs|G...,e9b6edc3-30c3-5b10-8fad-1648576874da,af61b489-9773-510d-bbeb-5c909fbf4068,['8a427f85-ea31-5abe-9819-de629a9d387c'],"lemma=da, pos=ADP"


In [13]:
save_dir = 'dataframes'

In [14]:
# reduced_df['group_str'] = reduced_df.apply(lambda row: f"LEMMA={row['lemma']}_POS={row['pos']}_PARTS_LEMMA={row['parts_lemma']}_PARTS_POS={row['parts_pos']}", axis=1)
# reduced_df['group_str'].value_counts()

In [ ]:
group_hashes = list(reduced_df['group_hash'].unique())
for group_hash in tqdm(group_hashes, total=len(group_hashes)):
    group_df = reduced_df[reduced_df['group_hash'] == group_hash].copy()
    if group_df['lemma'].nunique() != 1:
        print(len(group_df))
        print(group_df['group_str'].unique())
        print(group_df['group_hash'].unique())
    if group_df['pos'].nunique() != 1:
        print(len(group_df))
        print(group_df['group_str'].unique())
        print(group_df['group_hash'].unique())
    assert(group_df['lemma'].nunique() == 1)
    assert(group_df['pos'].nunique() == 1)
    pos = group_df.iloc[0]['pos']
    lemma = group_df.iloc[0]['lemma']

    for i, row in group_df.iterrows():
        sentence_hashes = eval(row['sentences'])
        assert(type(sentence_hashes) is list)
        for j in range(3):
            if len(sentence_hashes) > j:
                sentence_hash = sentence_hashes[j]
                sentence_it = sentence_df.loc[sentence_hash, 'text_it']
                sentence_en = sentence_df.loc[sentence_hash, 'text_en']
                group_df.loc[i, f'sentence_{j+1}_it'] = sentence_it
                group_df.loc[i, f'sentence_{j+1}_en'] = sentence_en
    group_df['translation_en'] = ''
    group_df.drop(columns=['sentences'], inplace=True)
    save_path = os.path.join(save_dir, pos, f"{lemma}.tsv")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    cols = [
        'text', 'lemma', 'pos', 'translation_en', 'token_count', 'token_pct', 'group_count',
        'group_pct', 'parts_lemma', 'parts_pos', 'xpos', 'deprel', 'feats', 'token_hash', 'group_hash',
        'group', 'sentence_1_it', 'sentence_1_en', 'sentence_2_it',
        'sentence_2_en', 'sentence_3_it', 'sentence_3_en'
    ]
    missing_cols = list(set(cols).difference(set(group_df.columns)))
    for missing_col in missing_cols:
        assert('sentence_' in missing_col)
        group_df[missing_col] = ''
    assert(set(cols) == set(group_df.columns))
    group_df = group_df[cols]
    for i, row in group_df.iterrows():
        for col in cols:
            if type(row[col]) == str:
                group_df.loc[i, col] = row[col].strip()

    temp = group_df[np.logical_and(
        group_df['pos'] == 'ADP',
        group_df['lemma'] == 'da'
    )]
    if len(temp) > 0:
        print(temp)
    group_df.to_csv(save_path, sep='\t', index=False)

100%|██████████| 200/200 [00:01<00:00, 110.94it/s]
